In [2]:
!pip install -q langchain langchain-groq

In [3]:
import os
from google.colab import userdata

# Get API key from Google Colab Secrets
api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets!")

os.environ["GROQ_API_KEY"] = api_key

from langchain_groq import ChatGroq

# Initialize the LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("TravelEase AI is ready!")

TravelEase AI is ready!


In [4]:
SYSTEM = """
ROLE:
You are TravelEase's friendly AI travel booking assistant.

BUSINESS:
TravelEase is a travel agency that helps customers
explore holiday packages and make booking enquiries.

AVAILABLE PACKAGES:
1. Goa - 3 Days - Rs 5000 per person
2. Kerala - 4 Days - Rs 8000 per person
3. Manali - 5 Days - Rs 12000 per person
4. Jaipur - 3 Days - Rs 6000 per person

BOOKING RULES:
1. All package prices are per person.
2. Customers must book at least 48 hours before travel.
3. Collect customer name, destination, travel date,
   and number of people for a booking enquiry.
4. Never confirm a booking without actual verification.
5. Never invent hotel, flight, or transport availability.
6. If information is unavailable, say:
   "I'll check with our travel team."

PRICING:
Calculate total package prices carefully.
Total price = Package price per person × Number of people.

RESPONSE STYLE:
- Be friendly and helpful.
- Keep answers under 50 words.
- Reply in English or Telugu according to the customer.
- Give clear and simple answers.
"""

In [5]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

# Create chain
chain = prompt | llm | StrOutputParser()

# Conversation memory
history = []

def ask(question: str) -> str:
    """
    Ask a question to TravelEase AI.
    Maintains conversation history.
    """

    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("Hi! What holiday packages do you offer?"))

Hi there! 🌞 We offer these holiday packages:

1. Goa – 3 days – ₹5,000 per person  
2. Kerala – 4 days – ₹8,000 per person  
3. Manali – 5 days – ₹12,000 per person  
4. Jaipur – 3 days – ₹6,000 per person  

Let me know which one interests you!


In [6]:
history = []

print("Customer:", "Hi, my name is Samhitha.")
print("TravelEase:", ask("Hi, my name is Samhitha."))

print("\nCustomer:", "What is my name?")
print("TravelEase:", ask("What is my name?"))

Customer: Hi, my name is Samhitha.
TravelEase: Hello Samhitha! 👋  
Could you let me know which destination you’d like, the travel date, and how many people will be traveling? I’ll calculate the total and check availability for you.

Customer: What is my name?
TravelEase: Your name is Samhitha.


In [7]:
import csv

test_cases = [
    {
        "question": "What is the price of the Goa package?",
        "must_include": "5000"
    },
    {
        "question": "How much is the Kerala package?",
        "must_include": "8000"
    },
    {
        "question": "Tell me the price of Manali package.",
        "must_include": "12000"
    },
    {
        "question": "What is the Jaipur package price?",
        "must_include": "6000"
    },
    {
        "question": "How much will two people pay for Goa?",
        "must_include": "10000"
    },
    {
        "question": "Can I book a trip with only 24 hours notice?",
        "must_include": "48 hours"
    },
    {
        "question": "What details do you need for a booking?",
        "must_include": "name"
    },
    {
        "question": "Do you provide hotel availability confirmation?",
        "must_include": "travel team"
    },
    {
        "question": "Are package prices per person?",
        "must_include": "per person"
    },
    {
        "question": "Do you offer international holiday packages?",
        "must_include": "travel team"
    }
]

with open("tests.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["question", "must_include"]
    )
    writer.writeheader()
    writer.writerows(test_cases)

print("tests.csv created successfully!")

tests.csv created successfully!


In [8]:
def run_tests(test_chain, path="tests.csv"):
    """
    Run customer questions and check expected keywords.
    """

    with open(path, "r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = test_chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower()
        success = expected in answer.lower()

        if success:
            passed += 1

        print(
            "PASS" if success else "FAIL",
            "|",
            row["question"],
            "->",
            answer[:100]
        )

    rate = passed / len(rows) * 100

    print(f"\nResult: {passed}/{len(rows)} passed ({rate:.0f}%)")

    return rate


baseline_score = run_tests(chain)

FAIL | What is the price of the Goa package? -> The Goa package is ₹5,000 per person. For a group, multiply ₹5,000 by the number of people. Let me k
FAIL | How much is the Kerala package? -> The Kerala package costs ₹8,000 per person.
FAIL | Tell me the price of Manali package. -> The Manali package costs ₹12,000 per person for the 5‑day stay.
FAIL | What is the Jaipur package price? -> The Jaipur package is ₹6,000 per person. If you’d like to book or need more details, just let me kno
FAIL | How much will two people pay for Goa? -> Two people for Goa will cost Rs 10,000 in total.
FAIL | Can I book a trip with only 24 hours notice? -> I’m sorry, but we require at least 48 hours’ notice before travel. If you’d like to check availabili
PASS | What details do you need for a booking? -> Sure! To start a booking, I’ll need:

1. Your full name  
2. Destination (Goa, Kerala, Manali, or Ja
PASS | Do you provide hotel availability confirmation? -> I’m not able to confirm hotel availability dire

In [9]:
SYSTEM_V2 = SYSTEM + """

ADDITIONAL IMPORTANT RULES:
- Always multiply the package price by the number of travellers.
- Clearly mention the total price when the number of travellers is given.
- If travel is less than 48 hours away, explain that the minimum
  booking notice is 48 hours.
- Never say a booking is successful unless verified.
"""

improved_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_V2),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

improved_chain = improved_prompt | llm | StrOutputParser()

improved_score = run_tests(improved_chain)

print("\nBaseline score:", baseline_score)
print("Improved score:", improved_score)

FAIL | What is the price of the Goa package? -> The Goa package is Rs 5,000 per person. If you let me know how many people, I can give you the total
FAIL | How much is the Kerala package? -> The Kerala package is ₹8,000 per person. If you let me know how many people, I can calculate the tot
FAIL | Tell me the price of Manali package. -> The Manali package is ₹12,000 per person. If you let me know how many people, I can calculate the to
FAIL | What is the Jaipur package price? -> The Jaipur package is Rs 6,000 per person for a 3‑day stay.
FAIL | How much will two people pay for Goa? -> Two people for Goa would cost ₹10,000 in total.
FAIL | Can I book a trip with only 24 hours notice? -> I’m sorry, but we require a minimum of 48 hours’ notice before travel. With only 24 hours left, we c
PASS | What details do you need for a booking? -> I’ll need a few details to start:  
1️⃣ Your name  
2️⃣ Destination (Goa, Kerala, Manali, Jaipur)  

PASS | Do you provide hotel availability confirmation

In [10]:
llm_temp1 = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

temp_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

temp_chain = temp_prompt | llm_temp1 | StrOutputParser()

print("Temperature 1.0 - Run 1")
score1 = run_tests(temp_chain)

print("\nTemperature 1.0 - Run 2")
score2 = run_tests(temp_chain)

print("\nComparison")
print("Run 1:", score1)
print("Run 2:", score2)

Temperature 1.0 - Run 1
FAIL | What is the price of the Goa package? -> The Goa 3‑day package costs **Rs 5,000 per person**.  
For example, for 2 people the total would be 
FAIL | How much is the Kerala package? -> The Kerala package is ₹8,000 per person. Let me know how many travelers you have, and I’ll calculate
FAIL | Tell me the price of Manali package. -> Sure! The Manali package costs **₹12,000 per person**. If you’d like a total estimate or more detail
PASS | What is the Jaipur package price? -> The Jaipur package is **Rs 6000 per person**.
FAIL | How much will two people pay for Goa? -> For Goa, two people will pay Rs 10,000 (₹5,000 × 2). Let me know if you'd like to book or need more 
FAIL | Can I book a trip with only 24 hours notice? -> I’m sorry, but we can only accept bookings at least 48 hours before the travel date. Let me know if 
PASS | What details do you need for a booking? -> For a booking I’ll need:

1. Your name  
2. Destination (Goa, Kerala, Manali, or Jaipur)  


In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "I want to visit Goa with 3 people. What is the total package price?"
        }
    ]
})

print(result["messages"][-1].content)

Sure! The Goa 3‑day package is ₹5,000 per person. For 3 people, the total cost is ₹15,000. Let me know if you’d like to proceed or need more details!


In [12]:
tasks = {
    "Calculate package price": "Plain Python Code",
    "Generate a travel advertisement": "One LLM Call",
    "Check package details, verify booking requirements, and process a booking enquiry": "Agent"
}

for task, category in tasks.items():
    print(task, "->", category)

Calculate package price -> Plain Python Code
Generate a travel advertisement -> One LLM Call
Check package details, verify booking requirements, and process a booking enquiry -> Agent


In [13]:
# Function 1: Get package price

def get_package_price(destination: str) -> int | None:
    """
    Return the package price per person for a destination.
    Return None if the destination is unavailable.
    """

    packages = {
        "goa": 5000,
        "kerala": 8000,
        "manali": 12000,
        "jaipur": 6000
    }

    return packages.get(destination.lower())


# Function 2: Check booking notice

def check_booking_notice(hours_before_travel: int) -> bool:
    """
    Check whether a booking meets the minimum 48-hour notice.
    Return True if booking notice is sufficient.
    """

    return hours_before_travel >= 48


# Test both functions

print("Goa package price:", get_package_price("Goa"))

print("Kerala package price:", get_package_price("Kerala"))

print("Booking with 72 hours notice:",
      check_booking_notice(72))

print("Booking with 24 hours notice:",
      check_booking_notice(24))

Goa package price: 5000
Kerala package price: 8000
Booking with 72 hours notice: True
Booking with 24 hours notice: False
